# Funnel theo tuần có cửa sổ chuyển đổi: con số PM đọc mỗi thứ Hai

PM của một cửa hàng online muốn một dòng trong báo cáo tuần: **"tuần trước, bao nhiêu % người vào xem đã thêm
giỏ và đã mua?"**. Module 03 đã dựng funnel có thứ tự bằng SQL trên cùng log Retailrocket (1 404 179 → 32 934 →
9 952 visitor). Bài này đi tiếp ở phía **đọc số theo thời gian** bằng pandas:

- Thời gian từ lần xem đầu tới lần mua đầu phân phối thế nào, và chọn **cửa sổ chuyển đổi** bao lâu.
- Tỉ lệ theo **tuần vào** có cửa sổ, kèm cờ tuần **đã chín** (đủ thời gian quan sát) để không báo một cú giảm giả.
- Funnel theo sản phẩm, để thấy chỗ hỏng nằm ở đâu và câu hỏi nào dữ liệu không trả lời được.

Dữ liệu: `data/raw/ecommerce-events/clean_events.csv` (Retailrocket, bản làm sạch; giấy phép gốc CC BY-NC-SA 4.0,
chỉ dùng để học, không đưa trích mẫu lên repo). Tải: `uv run python data/download_dataset.py ecommerce-events`.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/basic/01-funnel-window-weekly -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Chuẩn hoá log và mốc đầu tiên của từng bước

`to_events(raw) -> DataFrame`: từ bảng đọc thẳng CSV (cột `timestamp` là epoch **mili giây** UTC, `visitorid`,
`event`, `itemid`) trả bảng có đúng 4 cột theo thứ tự `visitor_id`, `event`, `item_id`, `event_time` (datetime).

`ordered_first_times(events) -> DataFrame`: mỗi dòng một visitor **có ít nhất một `view`**, index `visitor_id`,
ba cột theo thứ tự:

| Cột | Định nghĩa |
| --- | --- |
| `t_view` | thời điểm `view` sớm nhất |
| `t_cart` | `addtocart` sớm nhất **vào hoặc sau** `t_view`; không có thì `NaT` |
| `t_buy` | `transaction` sớm nhất **vào hoặc sau** `t_cart`; không có `t_cart` thì `NaT` |

Đây là funnel có thứ tự theo visitor, cùng quy tắc với module 03 (không xét mặt hàng).

In [ ]:
def to_events(raw: pd.DataFrame) -> pd.DataFrame:
    """visitor_id, event, item_id, event_time (từ epoch mili giây)."""
    return pd.DataFrame({"visitor_id": raw["visitorid"], "event": raw["event"], "item_id": raw["itemid"],
                         "event_time": pd.to_datetime(raw["timestamp"], unit="ms")})


def ordered_first_times(events: pd.DataFrame) -> pd.DataFrame:
    """Index visitor_id (visitor có view); cột t_view, t_cart, t_buy theo thứ tự bước."""
    t_view = events[events.event == "view"].groupby("visitor_id").event_time.min().rename("t_view")
    cart = events[events.event == "addtocart"].join(t_view, on="visitor_id", how="inner")
    t_cart = cart[cart.event_time >= cart.t_view].groupby("visitor_id").event_time.min().rename("t_cart")
    buy = events[events.event == "transaction"].join(t_cart, on="visitor_id", how="inner")
    t_buy = buy[buy.event_time >= buy.t_cart].groupby("visitor_id").event_time.min().rename("t_buy")
    out = pd.concat([t_view, t_cart, t_buy], axis=1)   # index theo t_view: mọi visitor có view
    out.index.name = "visitor_id"
    return out

## 2. Thời gian tới khi mua và funnel theo tuần

`conversion_within(ft, hours) -> Series`: trong số visitor có `t_buy`, tỉ lệ có `t_buy - t_view` **không quá**
`h` giờ, với mỗi `h` trong danh sách `hours`. Index là chính các giá trị `hours`, tên Series `share_of_buyers`.

`weekly_funnel(ft, window, data_end) -> DataFrame`: gom visitor theo **tuần của `t_view`** (tuần bắt đầu thứ Hai,
index là `Timestamp` 00:00 thứ Hai đó), bốn cột theo thứ tự:

- `visitors`: số visitor của tuần;
- `cart_rate`: tỉ lệ có `t_cart - t_view <= window`;
- `buy_rate`: tỉ lệ có `t_buy - t_view <= window`;
- `mature`: `True` khi `tuần + 7 ngày + window <= data_end`, tức visitor vào cuối tuần cũng đã có đủ cửa sổ.

Gợi ý: so sánh với `NaT` cho `False`, nên mean của cột so sánh là tỉ lệ cần tìm.

In [ ]:
def conversion_within(ft: pd.DataFrame, hours: list[float]) -> pd.Series:
    """Tỉ lệ người mua mua trong h giờ kể từ lần xem đầu, với mỗi h."""
    waited = (ft.t_buy - ft.t_view).dropna().dt.total_seconds() / 3600
    return pd.Series([(waited <= h).mean() for h in hours], index=list(hours), name="share_of_buyers")


def weekly_funnel(ft: pd.DataFrame, window: pd.Timedelta, data_end: pd.Timestamp) -> pd.DataFrame:
    """Theo tuần của t_view (bắt đầu thứ Hai): visitors, cart_rate, buy_rate, mature."""
    week = ft.t_view.dt.to_period("W-SUN").dt.start_time.rename("week")
    flags = pd.DataFrame({"cart": (ft.t_cart - ft.t_view) <= window,      # NaT so sánh ra False
                          "buy": (ft.t_buy - ft.t_view) <= window})
    out = flags.groupby(week).agg(visitors=("cart", "size"), cart_rate=("cart", "mean"), buy_rate=("buy", "mean"))
    out["mature"] = out.index + pd.Timedelta(days=7) + window <= data_end
    return out

In [ ]:
# explore: đọc log thật và dựng funnel có thứ tự
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
path = ROOT / "data" / "raw" / "ecommerce-events" / "clean_events.csv"
if not path.exists():
    raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py ecommerce-events")
events = to_events(pd.read_csv(path, usecols=["timestamp", "visitorid", "event", "itemid"]))
data_end = events.event_time.max()
print(f"{len(events):,} sự kiện, {events.visitor_id.nunique():,} visitor, "
      f"{events.event_time.min():%Y-%m-%d %H:%M} → {data_end:%Y-%m-%d %H:%M}")

ft = ordered_first_times(events)
print(ft.notna().sum().to_dict())

In [ ]:
# explore: thời gian từ lần xem đầu tới lần mua đầu (trục log)
hours = (ft.t_buy - ft.t_view).dropna().dt.total_seconds() / 3600
print("phân vị (giờ):", hours.quantile([0.25, 0.5, 0.75, 0.9]).round(1).to_dict(), "| mean:", round(hours.mean(), 1))
print(conversion_within(ft, [1, 24, 24 * 7, 24 * 30]).round(4).to_dict())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(hours.clip(lower=1 / 60), bins=np.logspace(np.log10(1 / 60), np.log10(24 * 140), 50), color="#4c78a8")
ax.set_xscale("log")
for h, label in [(1, "1 giờ"), (24, "1 ngày"), (24 * 7, "7 ngày"), (24 * 30, "30 ngày")]:
    ax.axvline(h, color="#888", lw=0.8, ls="--")
    ax.annotate(label, (h, ax.get_ylim()[1] * 0.9), fontsize=8, ha="left")
ax.set_xlabel("giờ từ lần xem đầu tới lần mua đầu (log)")
ax.set_ylabel("số người mua")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Phần lớn mua ngay trong phiên đầu, một đuôi dài mua sau nhiều ngày", loc="left")
plt.show()

In [ ]:
# explore: buy rate theo tuần với ba cửa sổ, và không có cửa sổ
curves = {f"{d} ngày": weekly_funnel(ft, pd.Timedelta(days=d), data_end) for d in (1, 7, 30)}
no_window = ft.groupby(ft.t_view.dt.to_period("W-SUN").dt.start_time).t_buy.apply(lambda s: s.notna().mean())
weekly = curves["7 ngày"]
print(weekly.round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 4))
for (label, w), color in zip(curves.items(), ["#9ecae9", "#4c78a8", "#1f3b5a"]):
    ok = w[w.mature]
    ax.plot(ok.index, ok.buy_rate * 100, marker="o", ms=3, color=color, label=f"cửa sổ {label} (tuần đã chín)")
    ax.plot(w.index[~w.mature], w.buy_rate[~w.mature] * 100, ls="none", marker="x", color=color)
ax.plot(no_window.index, no_window * 100, color="#e45756", ls="--", label="không cửa sổ (mua bất kỳ lúc nào)")
ax.set_ylabel("% visitor mua")
ax.set_xlabel("tuần của lần xem đầu (x: tuần chưa chín)")
ax.legend(frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Không có cửa sổ, tỉ lệ tự giảm về cuối kỳ dù không có gì thay đổi", loc="left")
plt.show()

In [ ]:
# explore: funnel theo sản phẩm (đếm visitor theo từng item, không xét thứ tự)
items = (events.groupby(["item_id", "event"]).visitor_id.nunique().unstack(fill_value=0)
         .reindex(columns=["view", "addtocart", "transaction"], fill_value=0))
items.columns = ["viewers", "carters", "buyers"]
popular = items[items.viewers >= 500].assign(cart_rate=lambda d: d.carters / d.viewers)
print(f"{len(popular)} sản phẩm có ≥ 500 người xem, {(popular.carters == 0).sum()} không ai thêm giỏ")
print(popular.sort_values("viewers", ascending=False).head(8).round(4).to_string())
print("sản phẩm có người mua nhiều hơn người thêm giỏ:", int((items.buyers > items.carters).sum()))

## Đọc số, rồi làm trắc nghiệm

1. **Chọn cửa sổ**: cửa sổ 7 ngày giữ bao nhiêu % người mua? Nếu PM đọc số mỗi thứ Hai, tuần nào là tuần gần nhất
   báo được?
2. **Tuần chưa chín**: tuần cuối có buy rate thấp nhất bảng. Có nên đặt nó vào chart cùng các tuần khác không?
3. **Tuần đầu**: tuần 27/04 có bao nhiêu ngày dữ liệu? "Lần xem đầu" của visitor tuần đó có chắc là lần xem đầu
   thật không (cắt trái)?
4. **Không có cửa sổ**: đường đỏ nét đứt kể câu chuyện gì, và vì sao câu chuyện đó sai?
5. **Theo sản phẩm**: sản phẩm được xem nhiều nhất gần như không ai thêm giỏ. Liệt kê ít nhất hai giả thuyết trước
   khi kết luận "trang sản phẩm tệ", và dữ liệu nào cần thêm vào tracking plan để kiểm.

Viết headline cho dòng "funnel tuần trước" mà PM sẽ đọc. Xong thì mở app, làm phần **trắc nghiệm** của bài.